In [3]:
from pathlib import Path 
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['MKL_SERVICE_FORCE_INTEL'] = '1'
from tqdm import tqdm
from IPython.display import Video

import torch
import numpy as np

import sys
sys.path.append("/srv/sferraro/choreographer/")

import envs
from envs.make import make

import matplotlib.pyplot as plt
import matplotlib.animation as animation

import mani_skill2
import mani_skill2.envs


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/elephant/miniconda3/envs/urlb/lib/python3.8/site-packages/robosuite-1.4.0-py3.8.egg/robosuite/scripts/setup_macros.py (__init__.py:9)


In [5]:
# Import agent model (WM + Actor Critic)
agent_path = Path(f'/srv/sferraro/choreographer/notebooks/object_pos_debug/models/intr/last_snapshot.pt')

def load_agent(agent_path):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    with agent_path.open('rb') as f:
        obj = torch.load(f, map_location=torch.device(device))
        agent = obj['agent']    
        step = obj['_global_step']
        agent.device = device
        agent.wm.device = device
        agent.wm.rssm.device = device
        agent.wm.rssm._cell.device = device
    return agent, step

agent, global_step = load_agent(agent_path)

In [6]:
from hydra import compose, initialize
from omegaconf import OmegaConf

initialize(config_path="../../exp_local/2023.05.11/114829_dreamer_obj_rsPanda_CustomLift_/.hydra", job_name="config")
cfg = compose(config_name="config")

In [7]:
# Agent parametrization
obs_type = agent.cfg.obs_type
action_repeat = agent.cfg.action_repeat
snapshot_ts = global_step * action_repeat

agent.reward_free = True

agent.use_selector = False
agent.detached_exploration = True

seed = agent.cfg.seed

task = "rsPanda_CustomLift"
domain = task.split("_")[0]

cfg.env.objects.minsize = 0.025

# Env creation
eval_env = make(task, obs_type, frame_stack=1, 
                    action_repeat=action_repeat, seed=seed, env_config=cfg.env)

ConfigAttributeError: Key 'task_reward' is not in struct
    full_key: env.task_reward
    object_type=dict

In [1]:
# get first observation from env (reset), feed it to encoder and get feat, feed features to actor for planning sequence, swap features to test out different objects
import utils
%matplotlib inline
from IPython import display
import imageio

newpath = r'./temp' 
if not os.path.exists(newpath):
    os.makedirs(newpath)

render_size = 64
camera = "agentview"

eval_mode = True

agent_state = None
meta = agent.init_meta()

dreamer_obs = eval_env.reset()
episodes = 1

step, episode, total_reward = 0, 0, 0

reward_fn = (
    lambda seq: agent.wm.heads["object_decoder"](
        seq["feat"], only_mlp=True
    )["objects_pos"][0]
    .mean[:, :, 1]
    .unsqueeze(-1)
) 
        
for ep in tqdm(range(episodes)):
    agent_state = None
    dreamer_obs = eval_env.reset()
    
    with torch.no_grad(), utils.eval_mode(agent):
        while not bool(dreamer_obs['is_last']):
            f, axs = plt.subplots(1, 2, figsize=(5, 5))
            action, agent_state = agent.act(
                                    dreamer_obs,
                                    meta,
                                    step,
                                    eval_mode=False,
                                    state=agent_state,
                                )
        
            dreamer_obs = eval_env.step(action)
        
            seg_pixels = np.argwhere(dreamer_obs["segmentation"][0])
            if len(seg_pixels) > 0:
                centroid = np.mean(seg_pixels, axis=0).astype(int)
                dreamer_obs["rgb"][:, centroid[0], centroid[1]] = [0,255,0]

            axs[0].imshow(dreamer_obs["rgb"].transpose(1,2,0))    
            axs[1].imshow(dreamer_obs["segmentation"][0])
            
            pos = dreamer_obs["objects_pos"]
            print(f"Cube position: {pos[0]}")
            # axs[1].set_title(f"{pos[0]}")
            
            feat = agent.wm.rssm.get_feat(agent_state[0]).unsqueeze(0)
            # print(
            predicted_pos = agent.wm.heads["object_decoder"](feat, only_mlp=True)["objects_pos"].mean
            
            print(f"Predicted position: {predicted_pos[0, 0].cpu().numpy()}")
            print(f"Pixels in view: { len(seg_pixels)}")
            
            contact = eval_env._env.check_contact(eval_env._env.robots[0].gripper, eval_env._env.cube)
            print(f"Contact: {contact}")
            
            plt.savefig(f'./temp/img_{step}.png', transparent = False, facecolor="white", bbox_inches='tight')
            plt.show()
        
            step += 1
    
    episode += 1
    
    frames = []
    
    for t in range(step):
        image = imageio.v2.imread(f'./temp/img_{t}.png')
        frames.append(image)
        
    imageio.mimsave('./out.gif', # output gif
                frames,          # array of input frames
                fps = 20)  


ModuleNotFoundError: No module named 'utils'